[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alanturin-g/Computational_Neuroscience_UNITO/blob/main/Perotti_5_EEG_load.ipynb)

In [ ]:
from google.colab import auth; auth.authenticate_user()
from google.colab import drive; drive.mount('/content/drive')

# EEG Eye State Classification

**Goal**: Predict whether a person's eyes are open or closed from EEG brain wave data.

**Dataset**: 14,980 EEG observations over 117 seconds
- 14 EEG sensors (AF3, F7, F3, FC5, T7, P7, O1, O2, P8, T8, FC6, F4, F8, AF4)
- Target: Eye state (0 = open, 1 = closed)
- Sampling rate: ~128 Hz (128 observations per second)

---

## Step 1: Import Libraries

In [1]:
# Data manipulation
import numpy as np
import pandas as pd
from scipy.io import arff

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('default')

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## Step 2: Load and Prepare Data

In [4]:
# A wrapper method to download files

import os, subprocess, shlex
import os
import subprocess

def download_from_drive(base_folder="."):
    file_name = 'eeg_dataframe.csv'
    file_ID = '1kLz9KSGsbA4rVc3Dn-U8hnLyfDizw1Zh'
    DEST_PATH = os.path.join(base_folder, file_name)
    if not os.path.exists(DEST_PATH):
        URL = f"https://drive.usercontent.google.com/download?id={file_ID}&confirm=t"
        cmd = ["wget", "-q", "--show-progress", "--progress=bar:force:noscroll",
            "--no-check-certificate", "-O", DEST_PATH, URL]
        ret = subprocess.call(cmd)  # shell=False by default
        if ret != 0:
            raise RuntimeError("Download failed. Check sharing settings or FILE_ID.")
        else:
            print("Downloaded " + file_name + "!")
    else:
        print(file_name + " already present:", DEST_PATH)

In [2]:
download_from_drive()
df = pd.read_csv('eeg_dataframe.csv')

# Define feature names
feature_names = [col for col in df.columns if col != 'eyeDetection']

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['eyeDetection'].value_counts().sort_index())
print(f"\nBalance: {df['eyeDetection'].value_counts(normalize=True).sort_index().values}")

df.head()

Dataset shape: (14980, 15)

Class distribution:
eyeDetection
0    8257
1    6723
Name: count, dtype: int64

Balance: [0.5512016 0.4487984]


,AF3,F7,F3,FC5,T7,P7,O1,O2,P8,T8,FC6,F4,F8,AF4,eyeDetection
0,4329.23,4009.23,4289.23,4148.21,4350.26,4586.15,4096.92,4641.03,4222.05,4238.46,4211.28,4280.51,4635.90,4393.85,0
1,4324.62,4004.62,4293.85,4148.72,4342.05,4586.67,4097.44,4638.97,4210.77,4226.67,4207.69,4279.49,4632.82,4384.10,0
2,4327.69,4006.67,4295.38,4156.41,4336.92,4583.59,4096.92,4630.26,4207.69,4222.05,4206.67,4282.05,4628.72,4389.23,0
3,4328.72,4011.79,4296.41,4155.90,4343.59,4582.56,4097.44,4630.77,4217.44,4235.38,4210.77,4287.69,4632.31,4396.41,0
4,4326.15,4011.79,4292.31,4151.28,4347.69,4586.67,4095.90,4627.69,4210.77,4244.10,4212.82,4288.21,4632.82,4398.46,0
